# Phase 8 — Inference smoke run on RWF-2000

This notebook runs the packaged clip-level inference on a small set of RWF-2000 validation videos. For each clip it writes a JSON verdict and an annotated MP4 showing person detections, ByteTrack IDs, short track trails, and the final clip verdict.

Attach the RWF-2000 dataset and your **Swin3D-T paired-view Kaggle Model**. The model must expose the project checkpoint (`best.pt` or an equivalent `.pt` file containing `architecture: swin3d_t` and `model_state_dict`). You do not need to attach the interaction-crops cache: inference runs detection and tracking on the clip. This smoke run uses one GPU; select T4 or T4x2, and enable Internet so the repo and YOLO11n weights can be fetched if needed. It does not train or evaluate model accuracy.


## 1. Clone the project and configure the inference runtime

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

import torch

REPO_URL = 'https://github.com/Ayaan-Ali-Khan/Violence-Detection.git'
REPO_DIR = Path('/kaggle/working/Violence-Detection')
GIT_REF = 'main'  # For a reproducible run, replace with the pushed commit SHA.

if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', '--all', '--tags'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', GIT_REF], check=True)
if GIT_REF in {'main', 'master'}:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', GIT_REF], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR / 'src'))
os.environ['OMP_NUM_THREADS'] = '2'
os.environ['MKL_NUM_THREADS'] = '2'
os.environ['OPENBLAS_NUM_THREADS'] = '1'

subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-kaggle.txt'], check=True)
print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Enable a Kaggle GPU accelerator before running inference.')
print('Using GPU 0:', torch.cuda.get_device_name(0))
print('Kaggle inputs:', *Path('/kaggle/input').iterdir(), sep='\n  ')

## 2. Find RWF-2000 and choose validation clips

In [ ]:
INPUT_ROOT = Path('/kaggle/input')
RWF_ROOT_OVERRIDE = None  # Set to the folder containing train/ and val/ if discovery is ambiguous.
SPLIT = 'train'
VIDEOS_PER_CLASS = 2

if RWF_ROOT_OVERRIDE is not None:
    RWF_ROOT = Path(RWF_ROOT_OVERRIDE)
else:
    rwf_roots = sorted({
        path.parent.parent for path in INPUT_ROOT.rglob('Fight')
        if path.is_dir()
        and path.parent.name == 'train'
        and (path.parent.parent / 'train' / 'NonFight').is_dir()
        and (path.parent.parent / 'val' / 'Fight').is_dir()
        and (path.parent.parent / 'val' / 'NonFight').is_dir()
    })
    print('RWF roots discovered:', *rwf_roots, sep='\n  ')
    if len(rwf_roots) != 1:
        raise RuntimeError('Attach exactly one RWF-2000 dataset or set RWF_ROOT_OVERRIDE.')
    RWF_ROOT = rwf_roots[0]

clip_items = []
for folder_name, ground_truth in [('Fight', 'Violent'), ('NonFight', 'NonViolent')]:
    class_dir = RWF_ROOT / SPLIT / folder_name
    clips = sorted(path for path in class_dir.iterdir() if path.is_file() and path.suffix.lower() == '.avi')
    if not clips:
        raise FileNotFoundError(f'No .avi clips found in {class_dir}')
    chosen = clips[:VIDEOS_PER_CLASS]
    clip_items.extend((clip, ground_truth) for clip in chosen)
    print(f'{ground_truth}: selected {len(chosen)} of {len(clips)} clips')
print('RWF root:', RWF_ROOT)
print('Selected clips:', *[f'{label}: {clip.name}' for clip, label in clip_items], sep='\n  ')

## 3. Locate and validate the attached Swin3D-T checkpoint

In [ ]:
# Optional: set the exact checkpoint path shown under /kaggle/input.
CHECKPOINT_PATH_OVERRIDE = None

def is_project_swin_checkpoint(path):
    try:
        checkpoint = torch.load(path, map_location='cpu', weights_only=False)
    except Exception as error:
        print(f'Skipping unreadable candidate {path}: {error}')
        return False
    valid = (
        isinstance(checkpoint, dict)
        and checkpoint.get('architecture') == 'swin3d_t'
        and isinstance(checkpoint.get('model_state_dict'), dict)
    )
    del checkpoint
    return valid

if CHECKPOINT_PATH_OVERRIDE is not None:
    CHECKPOINT_PATH = Path(CHECKPOINT_PATH_OVERRIDE)
    if not CHECKPOINT_PATH.is_file() or not is_project_swin_checkpoint(CHECKPOINT_PATH):
        raise ValueError(f'Not a compatible project Swin3D-T checkpoint: {CHECKPOINT_PATH}')
else:
    candidates = sorted({
        path for suffix in ('*.pt', '*.pth', '*.ckpt')
        for path in INPUT_ROOT.rglob(suffix)
        if 'swin' in str(path).lower() or path.name.lower() == 'best.pt'
    })
    preferred = [
        path for path in candidates
        if path.name.lower() == 'best.pt'
        and 'swin3d_t_rwf2000_paired_views' in str(path).lower()
    ]
    if not preferred:
        preferred = [
            path for path in candidates
            if 'swin' in str(path).lower() and path.name.lower() != 'last.pt'
        ]
    if preferred:
        candidates = preferred
    compatible = [path for path in candidates if is_project_swin_checkpoint(path)]
    print('Compatible Swin3D-T checkpoints:', *compatible, sep='\n  ')
    if len(compatible) != 1:
        raise RuntimeError(
            'Expected one compatible Swin3D-T checkpoint. Set CHECKPOINT_PATH_OVERRIDE '
            'to the attached best.pt file. Kaggle Models must expose the project .pt checkpoint.'
        )
    CHECKPOINT_PATH = compatible[0]
print('Using checkpoint:', CHECKPOINT_PATH)

## 4. Run inference and write JSON plus annotated MP4s

In [ ]:
import json
import pandas as pd
from violence_detection.inference import ViolencePredictor

OUTPUT_DIR = Path('/kaggle/working/phase8_inference')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = 'cuda:0'  # A single GPU is sufficient for clip inference.

predictor = ViolencePredictor(checkpoint=CHECKPOINT_PATH, device=DEVICE)
results = []
for clip, ground_truth in clip_items:
    safe_stem = clip.stem.replace(' ', '_')
    annotated_path = OUTPUT_DIR / f'{ground_truth}_{safe_stem}_annotated.mp4'
    result = predictor.predict(clip, output_video=annotated_path)
    record = {
        'ground_truth': ground_truth,
        **result,
    }
    json_path = OUTPUT_DIR / f'{ground_truth}_{safe_stem}_result.json'
    json_path.write_text(json.dumps(record, indent=2) + '\n', encoding='utf-8')
    results.append(record)
    print(f"{ground_truth} sample {clip.name}: {result['final_verdict']} "
          f"(P(violent)={result['violent_probability']:.3f}); video={annotated_path}")

summary_columns = [
    'ground_truth', 'final_verdict', 'violent_probability', 'nonviolent_probability',
    'verdict_confidence', 'unique_tracked_person_count', 'person_detection_count',
    'crop_fallback_rate', 'annotated_video',
]
summary = pd.DataFrame(results)[summary_columns]
summary_path = OUTPUT_DIR / 'inference_summary.csv'
summary.to_csv(summary_path, index=False)
display(summary)
print('Saved summary:', summary_path)

## 5. Preview annotated results and inspect downloadable outputs

The verdict is for the entire clip and is repeated in the video banner; it is not a frame-by-frame violence timeline. The ground-truth label shown in the summary is for smoke-run context only and is not passed to the model.

In [ ]:
from IPython.display import Video, display

for record in results:
    print(f"Ground truth: {record['ground_truth']} | Model verdict: {record['final_verdict']} | "
          f"P(violent): {record['violent_probability']:.3f}")
    display(Video(record['annotated_video'], embed=True, width=720))

print('Output files:')
for path in sorted(OUTPUT_DIR.iterdir()):
    if path.is_file():
        print(path.name, f'{path.stat().st_size / (1024 * 1024):.2f} MiB')

## Download

After saving a notebook version, download `/kaggle/working/phase8_inference/`. It contains one annotated MP4 and result JSON per selected clip, plus `inference_summary.csv`. The checkpoint is not copied into the output folder.